In [2]:
import sys
sys.path.append("..")
import ee
import pandas as pd
from src.config import GEE_PROJECT, AOI_BOX

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_BOX)

# same grid (cell_id pehle jaisi hi banegi)
def make_grid(cell_deg=0.01):
    xmin, ymin, xmax, ymax = AOI_BOX
    feats, i, x = [], 0, xmin
    while x < xmax - 1e-9:
        y = ymin
        while y < ymax - 1e-9:
            geom = ee.Geometry.Rectangle([x, y, x + cell_deg, y + cell_deg])
            feats.append(ee.Feature(geom, {"cell_id": i}))
            i += 1
            y += cell_deg
        x += cell_deg
    return ee.FeatureCollection(feats)

grid = make_grid()

def to_df(fc, chunk=2500):
    n = fc.size().getInfo()
    parts = []
    for start in range(0, n, chunk):
        sub = ee.FeatureCollection(fc.toList(chunk, start))
        feats = sub.getInfo()["features"]
        parts.append(pd.DataFrame([f["properties"] for f in feats]))
        print(f"fetched {start + len(feats)} / {n}")
    return pd.concat(parts, ignore_index=True)

srtm = ee.Image("USGS/SRTMGL1_003")
hand = ee.Image("MERIT/Hydro/v1_0_1").select("hnd").rename("hand")
elev_mean = srtm.focal_mean(1000, "circle", "meters").rename("elev_mean_1km")
elev_rel = srtm.subtract(elev_mean).rename("elev_rel")

wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
cropland = wc.eq(40).rename("cropland")
builtup = wc.eq(50).rename("builtup")
wetland = wc.eq(90).Or(wc.eq(95)).rename("wetland_mangrove")

new_img = hand.addBands([elev_mean, elev_rel, cropland, builtup, wetland])
new_fc = new_img.reduceRegions(collection=grid, reducer=ee.Reducer.mean(), scale=100, tileScale=4)
new_df = to_df(new_fc)
print(new_df.describe().round(2))

# purani CSV se jodo
old = pd.read_csv("../data/processed/features_2022.csv")
full = old.merge(new_df, on="cell_id")
full.to_csv("../data/processed/features_2022_v2.csv", index=False)
print(full.shape)

fetched 2500 / 5600
fetched 5000 / 5600
fetched 5600 / 5600
       builtup  cell_id  cropland  elev_mean_1km  elev_rel     hand  \
count  5600.00  5600.00   5600.00        5600.00   5600.00  4566.00   
mean      0.01  2799.50      0.54           5.48      0.00     1.23   
std       0.04  1616.73      0.36           5.15      0.99     3.07   
min       0.00     0.00      0.00           0.00    -28.37     0.00   
25%       0.00  1399.75      0.12           2.37     -0.17     0.48   
50%       0.00  2799.50      0.64           4.17      0.00     0.82   
75%       0.00  4199.25      0.86           8.09      0.14     1.45   
max       0.86  5599.00      1.00          88.75     39.93   112.33   

       wetland_mangrove  
count           5600.00  
mean               0.03  
std                0.14  
min                0.00  
25%                0.00  
50%                0.00  
75%                0.00  
max                1.00  
(5600, 16)
